# Mortgage Collateral Risk Deep-Learning Assistant

**Course:** Advanced Machine Learning — Final Project  
**Task:** Image Analysis with CNNs and Text Analysis with RNNs  
**Domain:** Real Estate / Mortgage Collateral Underwriting  

---

## Business Problem

A mortgage underwriter or P2P-lending platform receives a property listing (photo + description) as collateral for a loan application. Manually assessing every listing's visual quality and description accuracy is slow and inconsistent. This project builds two specialised deep-learning models that automate the assessment:

- **CNN branch** — classifies a property photo into a **price band** (Budget / Economy / Mid-Range / Premium / Luxury), indicating the visual quality tier of the collateral.
- **RNN/LSTM branch** — classifies a property listing description into the same price band, capturing textual signals of value (amenities, condition language, neighbourhood cues).
- **Business integration** — a decision rule combines both predictions against the applicant's *claimed* value band to flag collateral risk: `LOW` (auto-proceed), `MEDIUM` (secondary review), or `HIGH` (manual appraisal required).
- **Joint model (+10 bonus)** — a single network consuming both image and text features predicts the price band end-to-end, benchmarked against each unimodal model.

**End user:** Mortgage underwriter / risk officer.  
**FP cost:** Unnecessary manual appraisal (operational overhead).  
**FN cost:** Over-lending on over-valued collateral (credit loss).  

---

## Team Contribution Table

| Member | Sections |
|--------|----------|
| Raivo Strods | All sections (individual project) |

## 1. Setup and Reproducibility

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import random
import json
import re
import math
import io
import zipfile
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from PIL import Image
from wordcloud import WordCloud

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid', palette='colorblind', font_scale=1.1)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 120)

RANDOM_STATE = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'Torchvision: {torchvision.__version__}')
print('All imports successful.')

In [ ]:
DATA_DIR = Path('data')
IMAGE_DIR = DATA_DIR / 'images'
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

BAND_NAMES = ['Budget', 'Economy', 'Mid-Range', 'Premium', 'Luxury']
NUM_CLASSES = len(BAND_NAMES)
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0  # set to 2+ for multi-core / GPU machines

## 2. Data Loading

### Dataset strategy

We use **two real-estate datasets from the same domain** (permitted by the project brief — Section 4):

1. **Image dataset (CNN):** [House Prices and Images — SoCal](https://www.kaggle.com/datasets/ted8080/house-prices-and-images-socal) — ~15,000 frontal house photos with sale prices. Each image is the exterior/front view of a property.
2. **Text dataset (RNN):** [Real Estate Data London 2024](https://www.kaggle.com/datasets/kanchana1990/real-estate-data-london-2024) — property listings with rich HTML descriptions (avg. well above 15 words) and asking prices.

Both datasets are from the real-estate / property valuation domain. We derive the **same 5-class ordinal target** (price band from quantiles) independently for each dataset.

For the **joint model (+10 bonus)**, we use the Airbnb NYC dataset which contains both `picture_url` (downloadable property images) and `description` (rich text) with `price` for the same listing — providing naturally paired image+text+label samples.

---

**Instructions for downloading the datasets:**

Option A — Using the Kaggle CLI:
```bash
pip install kaggle
# Place your kaggle.json API token in ~/.kaggle/
kaggle datasets download -d ted8080/house-prices-and-images-socal -p data/
kaggle datasets download -d kanchana1990/real-estate-data-london-2024 -p data/
kaggle datasets download -d arianazmoudeh/airbnbopendata -p data/
```

Option B — Manual download from Kaggle and extract into the `data/` folder.

The cell below attempts the Kaggle CLI download. If the API token is not configured, download manually.

In [ ]:
# Attempt Kaggle download — skip gracefully if API key not configured
import subprocess, sys

datasets = {
    'ted8080/house-prices-and-images-socal': ['socal2.csv', 'socal2'],
    'kanchana1990/real-estate-data-london-2024': ['realestate_data_london_2024_nov.csv'],
    'arianazmoudeh/airbnbopendata': ['Airbnb_Open_Data.csv'],
}

for ds, markers in datasets.items():
    already = any((DATA_DIR / m).exists() for m in markers)
    if not already:
        try:
            subprocess.run(
                [sys.executable, '-m', 'kaggle', 'datasets', 'download',
                 '-d', ds, '-p', str(DATA_DIR), '--unzip'],
                check=True, capture_output=True, text=True, timeout=300
            )
            print(f'Downloaded: {ds}')
        except Exception as e:
            print(f'Could not auto-download {ds}: {e}')
            print('Please download manually from Kaggle and extract into data/')
    else:
        print(f'Already exists: {ds.split("/")[-1]} ({", ".join(markers)})')

print('\nContents of data/:')
for p in sorted(DATA_DIR.iterdir()):
    if p.name.startswith('.') or p.name == 'glove.6B.100d.txt':
        continue
    if p.is_dir():
        n = sum(1 for _ in p.rglob('*') if _.is_file())
        print(f'  {p.name}/ ({n} files)')
    else:
        print(f'  {p.name} ({p.stat().st_size / 1e6:.1f} MB)')

### 2.1 Load Image Dataset (SoCal Houses)

In [ ]:
# The SoCal dataset has images in folders and a CSV mapping image -> price
# Adapt path to actual extracted structure
socal_csv_candidates = list(DATA_DIR.rglob('*.csv'))
print('CSV files found:', [str(p) for p in socal_csv_candidates])

# Load the main CSV that maps images to prices
socal_csv = None
for f in socal_csv_candidates:
    df_tmp = pd.read_csv(f, nrows=5)
    cols_lower = [c.lower() for c in df_tmp.columns]
    if any('price' in c for c in cols_lower) and any('image' in c or 'img' in c or 'file' in c or 'path' in c for c in cols_lower):
        socal_csv = f
        break

if socal_csv is None:
    # Try finding image folders with numeric price-based folder names
    img_dirs = [d for d in DATA_DIR.rglob('*') if d.is_dir() and list(d.glob('*.jpg'))]
    print(f'Image directories found: {len(img_dirs)}')
    if img_dirs:
        print('Sample dirs:', [d.name for d in img_dirs[:5]])

# For SoCal dataset: images are typically in folders named by price or with a mapping CSV
# We'll handle both structures
all_image_files = sorted(DATA_DIR.rglob('*.jpg')) + sorted(DATA_DIR.rglob('*.jpeg')) + sorted(DATA_DIR.rglob('*.png'))
print(f'\nTotal image files found: {len(all_image_files)}')
if all_image_files:
    print(f'Sample paths: {[str(p) for p in all_image_files[:3]]}')

In [ ]:
# Build the image dataframe with file paths and prices
# The SoCal dataset has socal2.csv (image_id, street, citi, ..., price)
# and images in socal2/socal_pics/{image_id}.jpg

def build_image_dataframe(data_dir):
    """Build a DataFrame of (image_path, price) from the SoCal dataset."""
    records = []

    # Look for the SoCal CSV specifically (contains 'image_id' column)
    socal_csv = None
    for csv_file in sorted(data_dir.rglob('*.csv')):
        try:
            cols = pd.read_csv(csv_file, nrows=0).columns.tolist()
            if 'image_id' in [c.lower() for c in cols]:
                socal_csv = csv_file
                break
        except Exception:
            continue

    if socal_csv is not None:
        df = pd.read_csv(socal_csv)
        id_col = next(c for c in df.columns if c.lower() == 'image_id')
        price_col = next(c for c in df.columns if 'price' in c.lower())
        pic_dir = list(data_dir.rglob('socal_pics'))
        pic_dir = pic_dir[0] if pic_dir else data_dir
        for _, row in df.iterrows():
            img_path = pic_dir / f'{int(row[id_col])}.jpg'
            if img_path.exists():
                records.append({'image_path': str(img_path), 'price': float(row[price_col])})
        print(f'Loaded {len(records)} image-price pairs from {socal_csv.name}')
    else:
        # Fallback: match any CSV with price + id columns to images
        all_imgs = {p.stem: p for p in data_dir.rglob('*.jpg')}
        all_imgs.update({p.stem: p for p in data_dir.rglob('*.png')})
        for csv_file in sorted(data_dir.rglob('*.csv')):
            try:
                df = pd.read_csv(csv_file)
                price_col = next((c for c in df.columns if 'price' in c.lower()), None)
                id_col = next((c for c in df.columns if any(k in c.lower() for k in ['id', 'index', 'zpid', 'image'])), None)
                if price_col and id_col:
                    for _, row in df.iterrows():
                        key = str(int(row[id_col]) if pd.api.types.is_numeric_dtype(type(row[id_col])) else row[id_col])
                        if key in all_imgs:
                            records.append({'image_path': str(all_imgs[key]), 'price': float(row[price_col])})
                    if records:
                        print(f'Loaded {len(records)} image-price pairs from {csv_file.name}')
                        break
            except Exception:
                continue

    if not records:
        raise FileNotFoundError(
            'Could not build image-price mapping. '
            'Please check that the SoCal dataset is extracted correctly in data/'
        )

    return pd.DataFrame(records)

df_images = build_image_dataframe(DATA_DIR)
print(f'Image dataset: {len(df_images)} samples')
print(f'Price range: ${df_images["price"].min():,.0f} — ${df_images["price"].max():,.0f}')
df_images.head()

In [ ]:
# Create 5 price bands from quantiles
def assign_price_bands(df, price_col='price', n_bands=5):
    """Bin prices into ordinal bands using quantiles."""
    df = df.copy()
    df['price_band'], bin_edges = pd.qcut(
        df[price_col], q=n_bands, labels=BAND_NAMES, retbins=True, duplicates='drop'
    )
    df['price_band_id'] = df['price_band'].cat.codes
    print('Price band edges:')
    for i, name in enumerate(BAND_NAMES):
        if i < len(bin_edges) - 1:
            print(f'  {name}: ${bin_edges[i]:,.0f} — ${bin_edges[i+1]:,.0f}')
    return df, bin_edges

df_images, img_bin_edges = assign_price_bands(df_images)
print(f'\nClass distribution:\n{df_images["price_band"].value_counts().sort_index()}')

### 2.2 Load Text Dataset (Real Estate Descriptions)

In [ ]:
# Load the London Real Estate or Airbnb text dataset
def load_text_dataset(data_dir):
    """Load a text dataset with property descriptions and prices."""
    # Try London Real Estate 2024 first
    for csv_file in sorted(data_dir.rglob('*.csv')):
        try:
            df = pd.read_csv(csv_file, nrows=10)
            cols_lower = {c.lower(): c for c in df.columns}
            has_desc = any(k in ' '.join(cols_lower.keys()) for k in ['description', 'desc', 'summary', 'text'])
            has_price = any(k in ' '.join(cols_lower.keys()) for k in ['price'])
            if has_desc and has_price:
                df_full = pd.read_csv(csv_file)
                desc_col = next(c for c in df_full.columns if any(k in c.lower() for k in ['description', 'desc', 'summary']))
                price_col = next(c for c in df_full.columns if 'price' in c.lower())
                print(f'Loaded text data from {csv_file.name}: {len(df_full)} rows')
                print(f'Description column: {desc_col}')
                print(f'Price column: {price_col}')
                return df_full, desc_col, price_col
        except Exception:
            continue
    raise FileNotFoundError('No text dataset with descriptions + prices found in data/')

df_text_raw, TEXT_DESC_COL, TEXT_PRICE_COL = load_text_dataset(DATA_DIR)
print(f'\nRaw text dataset shape: {df_text_raw.shape}')
df_text_raw.head(3)

In [ ]:
# Clean the text dataset
def clean_text_data(df, desc_col, price_col):
    """Clean descriptions and prices."""
    df = df.copy()
    
    # Clean descriptions: remove HTML tags, normalise whitespace
    df[desc_col] = df[desc_col].astype(str)
    df[desc_col] = df[desc_col].str.replace(r'<[^>]+>', ' ', regex=True)  # strip HTML
    df[desc_col] = df[desc_col].str.replace(r'\s+', ' ', regex=True).str.strip()
    
    # Clean prices: remove currency symbols, commas, convert to float
    df[price_col] = df[price_col].astype(str)
    df[price_col] = df[price_col].str.replace(r'[£$€,\s]', '', regex=True)
    df[price_col] = pd.to_numeric(df[price_col], errors='coerce')
    
    # Drop rows with missing descriptions or prices
    before = len(df)
    df = df.dropna(subset=[desc_col, price_col])
    df = df[df[price_col] > 0]
    
    # Compute word count and filter to >= 15 words
    df['word_count'] = df[desc_col].str.split().str.len()
    df = df[df['word_count'] >= 15]
    
    # Remove extreme price outliers (below 1st and above 99th percentile)
    q01, q99 = df[price_col].quantile([0.01, 0.99])
    df = df[(df[price_col] >= q01) & (df[price_col] <= q99)]
    
    print(f'Cleaned: {before} -> {len(df)} samples (after dropping short texts, missing prices, outliers)')
    print(f'Word count stats: mean={df["word_count"].mean():.0f}, median={df["word_count"].median():.0f}, min={df["word_count"].min()}')
    return df.reset_index(drop=True)

df_text = clean_text_data(df_text_raw, TEXT_DESC_COL, TEXT_PRICE_COL)
df_text, txt_bin_edges = assign_price_bands(df_text, price_col=TEXT_PRICE_COL)
print(f'\nText dataset class distribution:\n{df_text["price_band"].value_counts().sort_index()}')

## 3. Exploratory Data Analysis (EDA)

### 3.1 Image EDA

In [ ]:
# Display sample images with labels — one row per price band, 4 images each
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(16, 4 * NUM_CLASSES))
fig.suptitle('Sample Property Images by Price Band', fontsize=16, fontweight='bold')

for row_axes, band in zip(axes, BAND_NAMES):
    subset = df_images[df_images['price_band'] == band]
    samples = subset.sample(min(4, len(subset)), random_state=RANDOM_STATE)
    for a, (_, row) in zip(row_axes, samples.iterrows()):
        try:
            img = Image.open(row['image_path']).convert('RGB')
            a.imshow(img)
            a.set_title(f"{row['price_band']}\n${row['price']:,.0f}", fontsize=9)
        except Exception:
            a.text(0.5, 0.5, 'Image\nnot found', ha='center', va='center', transform=a.transAxes)
        a.axis('off')
    for a in row_axes[len(samples):]:
        a.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Image dimension statistics
widths, heights, channels = [], [], []
sample_paths = df_images['image_path'].sample(min(500, len(df_images)), random_state=RANDOM_STATE)
for p in sample_paths:
    try:
        img = Image.open(p)
        w, h = img.size
        c = len(img.getbands())
        widths.append(w)
        heights.append(h)
        channels.append(c)
    except Exception:
        pass

print('=== Image Dimension Statistics (sampled 500) ===')
print(f'Width  — mean: {np.mean(widths):.0f}, std: {np.std(widths):.0f}, '
      f'min: {np.min(widths)}, max: {np.max(widths)}')
print(f'Height — mean: {np.mean(heights):.0f}, std: {np.std(heights):.0f}, '
      f'min: {np.min(heights)}, max: {np.max(heights)}')
print(f'Channels — {Counter(channels)}')

In [ ]:
# Class distribution bar chart — images
fig, ax = plt.subplots(figsize=(10, 5))
counts = df_images['price_band'].value_counts().reindex(BAND_NAMES)
counts.plot.bar(ax=ax, color=sns.color_palette('colorblind', NUM_CLASSES), edgecolor='black')
ax.set_title('Image Dataset: Price Band Distribution', fontweight='bold')
ax.set_xlabel('Price Band')
ax.set_ylabel('Count')
for i, v in enumerate(counts):
    ax.text(i, v + 10, str(v), ha='center', fontweight='bold')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 3.2 Text EDA

In [ ]:
# Text length distribution histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_text['word_count'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(df_text['word_count'].median(), color='red', linestyle='--', label=f'Median: {df_text["word_count"].median():.0f}')
axes[0].axvline(df_text['word_count'].mean(), color='orange', linestyle='--', label=f'Mean: {df_text["word_count"].mean():.0f}')
axes[0].set_title('Text Length Distribution (words)', fontweight='bold')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Class distribution bar chart — text
counts_txt = df_text['price_band'].value_counts().reindex(BAND_NAMES)
counts_txt.plot.bar(ax=axes[1], color=sns.color_palette('colorblind', NUM_CLASSES), edgecolor='black')
axes[1].set_title('Text Dataset: Price Band Distribution', fontweight='bold')
axes[1].set_xlabel('Price Band')
axes[1].set_ylabel('Count')
for i, v in enumerate(counts_txt):
    axes[1].text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(f'Text dataset: {len(df_text)} samples')
print(f'Word count — mean: {df_text["word_count"].mean():.0f}, median: {df_text["word_count"].median():.0f}, '
      f'min: {df_text["word_count"].min()}, max: {df_text["word_count"].max()}')

In [ ]:
# Word cloud per class
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(25, 5))
fig.suptitle('Word Clouds by Price Band (Text Descriptions)', fontsize=14, fontweight='bold')

for ax, band in zip(axes, BAND_NAMES):
    subset = df_text[df_text['price_band'] == band]
    text = ' '.join(subset[TEXT_DESC_COL].values)
    wc = WordCloud(width=400, height=300, background_color='white',
                   max_words=80, colormap='viridis').generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(band, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Top-20 most frequent terms per class
from collections import Counter
import string

STOP_WORDS = set('the a an and or is are was were be been being have has had '
                 'do does did will would shall should may might can could '
                 'for of to in on at by with from this that these those it its '
                 'i we you he she they me him her us them my your his our their '
                 'all each every both few many much some any no not so as than '
                 'if but about into through during before after above below '
                 'between out off over under again further then once also just '
                 'very own same up down here there when where how what which who '
                 'whom why s t d m re ve ll'.split())

print('Top-15 terms per price band:\n')
for band in BAND_NAMES:
    subset = df_text[df_text['price_band'] == band]
    words = ' '.join(subset[TEXT_DESC_COL].values).lower()
    words = words.translate(str.maketrans('', '', string.punctuation))
    tokens = [w for w in words.split() if w not in STOP_WORDS and len(w) > 2]
    top = Counter(tokens).most_common(15)
    print(f'{band}: {[w for w, _ in top]}')

In [ ]:
# Representative text samples per class (3 per band)
print('=== Representative Text Samples ===\n')
for band in BAND_NAMES:
    subset = df_text[df_text['price_band'] == band]
    samples = subset.sample(min(3, len(subset)), random_state=RANDOM_STATE)
    print(f'--- {band} ---')
    for _, row in samples.iterrows():
        desc = row[TEXT_DESC_COL][:200] + ('...' if len(row[TEXT_DESC_COL]) > 200 else '')
        print(f'  [{row[TEXT_PRICE_COL]:,.0f}] {desc}')
    print()

## 4. Data Preprocessing

### 4.1 Image Preprocessing

In [ ]:
# ImageNet normalisation stats (for transfer learning compatibility)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Training augmentation pipeline:')
print('  - Resize to 224x224')
print('  - Random horizontal flip (p=0.5)')
print('  - Random rotation (±10°)')
print('  - Colour jitter (brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05)')
print('  - Normalise with ImageNet mean/std')
print()
print('Validation/test pipeline:')
print('  - Resize to 224x224')
print('  - Normalise with ImageNet mean/std')

In [ ]:
class PropertyImageDataset(Dataset):
    """PyTorch Dataset for property images with price band labels."""
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = int(row['price_band_id'])
        return img, label

### 4.2 Text Preprocessing

In [ ]:
# Tokenisation and vocabulary building
def tokenize(text):
    """Simple word-level tokeniser: lowercase, strip punctuation, split on whitespace."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

# Build vocabulary from training data (will be rebuilt after split)
all_tokens = []
for desc in df_text[TEXT_DESC_COL]:
    all_tokens.extend(tokenize(desc))

token_counts = Counter(all_tokens)
print(f'Total tokens: {len(all_tokens):,}')
print(f'Unique tokens: {len(token_counts):,}')
print(f'Top 20: {token_counts.most_common(20)}')

In [ ]:
# Determine max sequence length from distribution (target <= 5% truncation)
token_lengths = df_text[TEXT_DESC_COL].apply(lambda x: len(tokenize(x)))

percentiles = [50, 75, 90, 95, 97, 99]
print('Token length percentiles:')
for p in percentiles:
    val = np.percentile(token_lengths, p)
    print(f'  {p}th: {val:.0f} tokens')

MAX_SEQ_LEN = int(np.percentile(token_lengths, 95))
pct_truncated = (token_lengths > MAX_SEQ_LEN).mean() * 100
print(f'\nChosen MAX_SEQ_LEN = {MAX_SEQ_LEN} (truncates {pct_truncated:.1f}% of texts)')

In [ ]:
class Vocabulary:
    """Word-level vocabulary with special tokens."""
    PAD = '<PAD>'
    UNK = '<UNK>'
    
    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.word2idx = {self.PAD: 0, self.UNK: 1}
        self.idx2word = {0: self.PAD, 1: self.UNK}
        self.word_freq = Counter()
    
    def build(self, texts):
        for text in texts:
            self.word_freq.update(tokenize(text))
        for word, freq in self.word_freq.items():
            if freq >= self.min_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word
        print(f'Vocabulary size: {len(self.word2idx)} (min_freq={self.min_freq})')
        return self
    
    def encode(self, text, max_len):
        tokens = tokenize(text)
        indices = [self.word2idx.get(t, self.word2idx[self.UNK]) for t in tokens[:max_len]]
        # Pad to max_len
        padding = [self.word2idx[self.PAD]] * (max_len - len(indices))
        return indices + padding
    
    def __len__(self):
        return len(self.word2idx)

In [ ]:
class PropertyTextDataset(Dataset):
    """PyTorch Dataset for property text descriptions with price band labels."""
    def __init__(self, df, desc_col, vocab, max_len):
        self.descriptions = df[desc_col].values
        self.labels = df['price_band_id'].values
        self.vocab = vocab
        self.max_len = max_len
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        encoded = self.vocab.encode(self.descriptions[idx], self.max_len)
        return torch.tensor(encoded, dtype=torch.long), int(self.labels[idx])

## 5. Train–Validation–Test Split

Stratified 70/15/15 split by price band with fixed seed for reproducibility. The test set is locked until final evaluation.

In [ ]:
# Image splits
df_img_train, df_img_temp = train_test_split(
    df_images, test_size=0.30, stratify=df_images['price_band_id'], random_state=RANDOM_STATE
)
df_img_val, df_img_test = train_test_split(
    df_img_temp, test_size=0.50, stratify=df_img_temp['price_band_id'], random_state=RANDOM_STATE
)

# Subsample for CPU — remove or increase MAX_TRAIN_SAMPLES on GPU
MAX_TRAIN_SAMPLES = 2000 if DEVICE.type == 'cpu' else len(df_img_train)
if len(df_img_train) > MAX_TRAIN_SAMPLES:
    per_band = MAX_TRAIN_SAMPLES // NUM_CLASSES
    df_img_train = pd.concat([
        g.sample(min(len(g), per_band), random_state=RANDOM_STATE)
        for _, g in df_img_train.groupby('price_band_id')
    ]).reset_index(drop=True)

print('=== Image Split ===')
print(f'Train: {len(df_img_train)} ({len(df_img_train)/len(df_images)*100:.0f}%)')
print(f'Val:   {len(df_img_val)} ({len(df_img_val)/len(df_images)*100:.0f}%)')
print(f'Test:  {len(df_img_test)} ({len(df_img_test)/len(df_images)*100:.0f}%)')
print(f'Train class distribution: {dict(df_img_train["price_band"].value_counts().sort_index())}')

In [ ]:
# Text splits
df_txt_train, df_txt_temp = train_test_split(
    df_text, test_size=0.30, stratify=df_text['price_band_id'], random_state=RANDOM_STATE
)
df_txt_val, df_txt_test = train_test_split(
    df_txt_temp, test_size=0.50, stratify=df_txt_temp['price_band_id'], random_state=RANDOM_STATE
)

print('=== Text Split ===')
print(f'Train: {len(df_txt_train)} ({len(df_txt_train)/len(df_text)*100:.0f}%)')
print(f'Val:   {len(df_txt_val)} ({len(df_txt_val)/len(df_text)*100:.0f}%)')
print(f'Test:  {len(df_txt_test)} ({len(df_txt_test)/len(df_text)*100:.0f}%)')
print(f'Train class distribution: {dict(df_txt_train["price_band"].value_counts().sort_index())}')

In [ ]:
# Build vocabulary from TRAINING text only
vocab = Vocabulary(min_freq=2)
vocab.build(df_txt_train[TEXT_DESC_COL])

# Create PyTorch datasets
img_train_ds = PropertyImageDataset(df_img_train, transform=train_transform)
img_val_ds = PropertyImageDataset(df_img_val, transform=val_test_transform)
img_test_ds = PropertyImageDataset(df_img_test, transform=val_test_transform)

txt_train_ds = PropertyTextDataset(df_txt_train, TEXT_DESC_COL, vocab, MAX_SEQ_LEN)
txt_val_ds = PropertyTextDataset(df_txt_val, TEXT_DESC_COL, vocab, MAX_SEQ_LEN)
txt_test_ds = PropertyTextDataset(df_txt_test, TEXT_DESC_COL, vocab, MAX_SEQ_LEN)

# Class-weighted sampler for imbalanced training sets
def make_weighted_sampler(labels):
    class_counts = Counter(labels)
    weights = [1.0 / class_counts[l] for l in labels]
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

img_train_loader = DataLoader(img_train_ds, batch_size=BATCH_SIZE, 
                              sampler=make_weighted_sampler(df_img_train['price_band_id'].tolist()),
                              num_workers=NUM_WORKERS, pin_memory=True)
img_val_loader = DataLoader(img_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
img_test_loader = DataLoader(img_test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

txt_train_loader = DataLoader(txt_train_ds, batch_size=BATCH_SIZE,
                              sampler=make_weighted_sampler(df_txt_train['price_band_id'].tolist()),
                              num_workers=NUM_WORKERS, pin_memory=True)
txt_val_loader = DataLoader(txt_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
txt_test_loader = DataLoader(txt_test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

print(f'Image loaders — train: {len(img_train_loader)} batches, val: {len(img_val_loader)}, test: {len(img_test_loader)}')
print(f'Text loaders  — train: {len(txt_train_loader)} batches, val: {len(txt_val_loader)}, test: {len(txt_test_loader)}')

## 6. CNN Branch (Image Model)

We implement and compare two CNN approaches as required:
1. **CNN from scratch** — custom architecture with 3+ convolutional blocks
2. **Transfer learning** — ResNet-50 pretrained on ImageNet, with frozen backbone and fine-tuned classification head

### 6.1 Training Utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.long().to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.long().to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return running_loss / total, correct / total, np.array(all_preds), np.array(all_labels)


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                device, epochs=25, patience=5, model_name='model'):
    """Full training loop with early stopping and best-model checkpointing."""
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_f1 = 0.0
    patience_counter = 0
    best_path = MODEL_DIR / f'{model_name}_best.pt'
    
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, val_preds, val_labels = evaluate(model, val_loader, criterion, device)
        val_f1 = f1_score(val_labels, val_preds, average='macro')
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        if scheduler:
            scheduler.step(val_loss)
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_path)
            patience_counter = 0
            marker = ' *'
        else:
            patience_counter += 1
            marker = ''
        
        if (epoch + 1) % 2 == 0 or epoch == 0 or marker:
            print(f'Epoch {epoch+1:3d}/{epochs} | '
                  f'Train loss: {train_loss:.4f} acc: {train_acc:.4f} | '
                  f'Val loss: {val_loss:.4f} acc: {val_acc:.4f} F1: {val_f1:.4f}{marker}')
        
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break
    
    model.load_state_dict(torch.load(best_path, weights_only=True))
    print(f'Best validation macro-F1: {best_val_f1:.4f}')
    return model, history


def plot_training_curves(history, title='Training Curves'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(history['train_loss']) + 1)
    
    axes[0].plot(epochs, history['train_loss'], 'o-', label='Train', markersize=3)
    axes[0].plot(epochs, history['val_loss'], 's-', label='Validation', markersize=3)
    axes[0].set_title(f'{title} — Loss', fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    
    axes[1].plot(epochs, history['train_acc'], 'o-', label='Train', markersize=3)
    axes[1].plot(epochs, history['val_acc'], 's-', label='Validation', markersize=3)
    axes[1].set_title(f'{title} — Accuracy', fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()


def evaluate_and_report(model, test_loader, criterion, device, class_names, title='Model'):
    """Full test-set evaluation: accuracy, F1, classification report, confusion matrix."""
    test_loss, test_acc, preds, labels = evaluate(model, test_loader, criterion, device)
    test_f1 = f1_score(labels, preds, average='macro')
    
    print(f'\n=== {title} — Test Results ===')
    print(f'Loss: {test_loss:.4f} | Accuracy: {test_acc:.4f} | Macro F1: {test_f1:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(labels, preds, target_names=class_names, digits=4))
    
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    ax.set_title(f'{title} — Confusion Matrix', fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return {'accuracy': test_acc, 'macro_f1': test_f1, 'loss': test_loss,
            'preds': preds, 'labels': labels}

### 6.2 CNN from Scratch

In [ ]:
class CNNFromScratch(nn.Module):
    """Custom CNN with 4 convolutional blocks + global average pooling."""
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 3 -> 32
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
            
            # Block 2: 32 -> 64
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
            
            # Block 3: 64 -> 128
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
            
            # Block 4: 128 -> 256
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

cnn_scratch = CNNFromScratch().to(DEVICE)
total_params = sum(p.numel() for p in cnn_scratch.parameters())
trainable_params = sum(p.numel() for p in cnn_scratch.parameters() if p.requires_grad)
print(f'CNN from scratch — Total params: {total_params:,} | Trainable: {trainable_params:,}')

In [ ]:
# Train CNN from scratch
criterion_cnn = nn.CrossEntropyLoss()
optimizer_scratch = optim.Adam(cnn_scratch.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_scratch = optim.lr_scheduler.ReduceLROnPlateau(optimizer_scratch, mode='min', factor=0.5, patience=3)

cnn_scratch, history_scratch = train_model(
    cnn_scratch, img_train_loader, img_val_loader, criterion_cnn, optimizer_scratch,
    scheduler_scratch, DEVICE, epochs=5, patience=3, model_name='cnn_scratch'
)

plot_training_curves(history_scratch, title='CNN From Scratch')

In [ ]:
# Test evaluation — CNN from scratch
results_cnn_scratch = evaluate_and_report(
    cnn_scratch, img_test_loader, criterion_cnn, DEVICE, BAND_NAMES, 'CNN From Scratch'
)

### 6.3 Transfer Learning — ResNet-50

In [ ]:
def build_resnet50_transfer(num_classes, freeze_backbone=True):
    """ResNet-50 with ImageNet weights, custom classification head."""
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.4),
        nn.Linear(256, num_classes),
    )
    return model

# Phase 1: frozen backbone
cnn_transfer = build_resnet50_transfer(NUM_CLASSES, freeze_backbone=True).to(DEVICE)
trainable = sum(p.numel() for p in cnn_transfer.parameters() if p.requires_grad)
total = sum(p.numel() for p in cnn_transfer.parameters())
print(f'ResNet-50 (frozen) — Total: {total:,} | Trainable: {trainable:,}')

In [ ]:
# Phase 1: Train only the classification head
optimizer_tf1 = optim.Adam(cnn_transfer.fc.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_tf1 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_tf1, mode='min', factor=0.5, patience=2)

print('=== Phase 1: Frozen backbone, training classification head ===')
cnn_transfer, history_tf_phase1 = train_model(
    cnn_transfer, img_train_loader, img_val_loader, criterion_cnn, optimizer_tf1,
    scheduler_tf1, DEVICE, epochs=3, patience=2, model_name='cnn_transfer_p1'
)

In [ ]:
# Phase 2: Unfreeze last two residual blocks (layer3, layer4) and fine-tune
for name, param in cnn_transfer.named_parameters():
    if 'layer3' in name or 'layer4' in name or 'fc' in name:
        param.requires_grad = True

trainable_ft = sum(p.numel() for p in cnn_transfer.parameters() if p.requires_grad)
print(f'Fine-tuning: {trainable_ft:,} trainable parameters')

optimizer_tf2 = optim.Adam(
    filter(lambda p: p.requires_grad, cnn_transfer.parameters()),
    lr=1e-4, weight_decay=1e-4
)
scheduler_tf2 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_tf2, mode='min', factor=0.5, patience=2)

print('\n=== Phase 2: Fine-tuning layer3 + layer4 + fc ===')
cnn_transfer, history_tf_phase2 = train_model(
    cnn_transfer, img_train_loader, img_val_loader, criterion_cnn, optimizer_tf2,
    scheduler_tf2, DEVICE, epochs=3, patience=2, model_name='cnn_transfer_p2'
)

# Combined curves
history_tf = {k: history_tf_phase1[k] + history_tf_phase2[k] for k in history_tf_phase1}
plot_training_curves(history_tf, title='ResNet-50 Transfer Learning (Phase 1 + 2)')

In [ ]:
# Test evaluation — Transfer learning
results_cnn_transfer = evaluate_and_report(
    cnn_transfer, img_test_loader, criterion_cnn, DEVICE, BAND_NAMES, 'ResNet-50 Transfer Learning'
)

### 6.4 CNN Comparison Discussion

In [ ]:
# Side-by-side comparison
cnn_comparison = pd.DataFrame({
    'Model': ['CNN From Scratch', 'ResNet-50 Transfer'],
    'Test Accuracy': [results_cnn_scratch['accuracy'], results_cnn_transfer['accuracy']],
    'Test Macro F1': [results_cnn_scratch['macro_f1'], results_cnn_transfer['macro_f1']],
    'Test Loss': [results_cnn_scratch['loss'], results_cnn_transfer['loss']],
})
print(cnn_comparison.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(2)
width = 0.35
ax.bar(x - width/2, cnn_comparison['Test Accuracy'], width, label='Accuracy')
ax.bar(x + width/2, cnn_comparison['Test Macro F1'], width, label='Macro F1')
ax.set_xticks(x)
ax.set_xticklabels(cnn_comparison['Model'])
ax.set_ylabel('Score')
ax.set_title('CNN: From Scratch vs Transfer Learning', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1)
for i, (acc, f1) in enumerate(zip(cnn_comparison['Test Accuracy'], cnn_comparison['Test Macro F1'])):
    ax.text(i - width/2, acc + 0.01, f'{acc:.3f}', ha='center', fontsize=9)
    ax.text(i + width/2, f1 + 0.01, f'{f1:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

**Discussion — CNN Comparison**

The transfer-learning approach (ResNet-50, ImageNet-pretrained) is expected to significantly outperform the from-scratch CNN for several reasons:

1. **Pre-learned feature hierarchy**: ResNet-50's convolutional filters, trained on 1.2 million ImageNet images, already capture edges, textures, and spatial patterns that generalise well to house photos.
2. **Data efficiency**: With ~10k training images our scratch model has limited capacity to learn robust features, while the transfer model only needs to adapt the final layers.
3. **Regularisation**: The frozen backbone acts as a strong regulariser, preventing overfitting common in smaller datasets.
4. **Domain similarity**: Exterior house photos share visual concepts (objects, structures, lighting) with ImageNet natural scenes, making transfer effective.

The two-phase fine-tuning strategy (freeze → unfreeze top blocks) gives an additional boost by adapting mid-level features to the specific property aesthetic while retaining low-level filters. We select the ResNet-50 transfer model as the CNN branch for downstream business integration.

## 7. RNN/LSTM Branch (Text Model)

We implement two architectural variations:
1. **Single-layer LSTM** with learned word embeddings
2. **Bidirectional stacked LSTM** with GloVe pre-trained embeddings and dropout

### 7.1 GloVe Embeddings (optional pre-trained)

In [ ]:
# Load GloVe embeddings (download 6B.100d if not already present)
GLOVE_DIM = 100
GLOVE_PATH = DATA_DIR / 'glove.6B.100d.txt'

def load_glove(path, vocab, dim=100):
    """Load pre-trained GloVe vectors and create embedding matrix for our vocabulary."""
    embeddings_index = {}
    if not path.exists():
        print(f'GloVe file not found at {path}')
        print('Download from https://nlp.stanford.edu/data/glove.6B.zip and extract glove.6B.100d.txt to data/')
        print('Falling back to random initialisation for "pre-trained" variation.')
        return None
    
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            if word in vocab.word2idx:
                embeddings_index[word] = np.array(values[1:], dtype='float32')
    
    embedding_matrix = np.random.normal(scale=0.6, size=(len(vocab), dim)).astype('float32')
    embedding_matrix[vocab.word2idx[Vocabulary.PAD]] = 0.0
    
    found = 0
    for word, idx in vocab.word2idx.items():
        if word in embeddings_index:
            embedding_matrix[idx] = embeddings_index[word]
            found += 1
    
    print(f'GloVe coverage: {found}/{len(vocab)} ({found/len(vocab)*100:.1f}%)')
    return torch.FloatTensor(embedding_matrix)

glove_matrix = load_glove(GLOVE_PATH, vocab, GLOVE_DIM)

### 7.2 LSTM Models

In [ ]:
class LSTMClassifier(nn.Module):
    """LSTM text classifier with configurable architecture."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers=1,
                 bidirectional=False, dropout=0.3, pretrained_embeddings=None):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if pretrained_embeddings is not None:
            self.embedding = nn.Embedding.from_pretrained(
                pretrained_embeddings, freeze=False, padding_idx=0
            )
        
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=bidirectional, dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * self.num_directions, num_classes)
    
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        # Pack padded sequences for efficiency
        lengths = (x != 0).sum(dim=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.clamp(min=1), batch_first=True, enforce_sorted=False
        )
        _, (hidden, _) = self.lstm(packed)
        
        if self.bidirectional:
            hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            hidden = hidden[-1]
        
        out = self.dropout(hidden)
        return self.fc(out)
    
    def get_features(self, x):
        """Extract the final hidden state as feature vector (for joint model)."""
        embedded = self.embedding(x)
        lengths = (x != 0).sum(dim=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.clamp(min=1), batch_first=True, enforce_sorted=False
        )
        _, (hidden, _) = self.lstm(packed)
        if self.bidirectional:
            return torch.cat((hidden[-2], hidden[-1]), dim=1)
        return hidden[-1]

In [ ]:
# Text training utilities
def train_text_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = torch.tensor(labels, dtype=torch.long).to(device) if not isinstance(labels, torch.Tensor) else labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate_text(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = torch.tensor(labels, dtype=torch.long).to(device) if not isinstance(labels, torch.Tensor) else labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return running_loss / total, correct / total, np.array(all_preds), np.array(all_labels)


def train_text_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                     device, epochs=25, patience=5, model_name='model'):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_f1 = 0.0
    patience_counter = 0
    best_path = MODEL_DIR / f'{model_name}_best.pt'
    
    for epoch in range(epochs):
        train_loss, train_acc = train_text_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, val_preds, val_labels = evaluate_text(model, val_loader, criterion, device)
        val_f1 = f1_score(val_labels, val_preds, average='macro')
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        if scheduler:
            scheduler.step(val_loss)
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_path)
            patience_counter = 0
            marker = ' *'
        else:
            patience_counter += 1
            marker = ''
        
        if (epoch + 1) % 2 == 0 or epoch == 0 or marker:
            print(f'Epoch {epoch+1:3d}/{epochs} | '
                  f'Train loss: {train_loss:.4f} acc: {train_acc:.4f} | '
                  f'Val loss: {val_loss:.4f} acc: {val_acc:.4f} F1: {val_f1:.4f}{marker}')
        
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break
    
    model.load_state_dict(torch.load(best_path, weights_only=True))
    print(f'Best validation macro-F1: {best_val_f1:.4f}')
    return model, history


def evaluate_text_and_report(model, test_loader, criterion, device, class_names, title='Model'):
    test_loss, test_acc, preds, labels = evaluate_text(model, test_loader, criterion, device)
    test_f1 = f1_score(labels, preds, average='macro')
    
    print(f'\n=== {title} — Test Results ===')
    print(f'Loss: {test_loss:.4f} | Accuracy: {test_acc:.4f} | Macro F1: {test_f1:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(labels, preds, target_names=class_names, digits=4))
    
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, cmap='Oranges', values_format='d')
    ax.set_title(f'{title} — Confusion Matrix', fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return {'accuracy': test_acc, 'macro_f1': test_f1, 'loss': test_loss,
            'preds': preds, 'labels': labels}

### 7.3 Variation A: Single-Layer LSTM with Learned Embeddings

In [ ]:
EMBED_DIM = 128
HIDDEN_DIM = 128

lstm_v1 = LSTMClassifier(
    vocab_size=len(vocab), embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES, num_layers=1, bidirectional=False, dropout=0.3
).to(DEVICE)

params_v1 = sum(p.numel() for p in lstm_v1.parameters())
print(f'LSTM v1 (single, learned embed) — Parameters: {params_v1:,}')

criterion_rnn = nn.CrossEntropyLoss()
optimizer_v1 = optim.Adam(lstm_v1.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler_v1 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_v1, mode='min', factor=0.5, patience=2)

lstm_v1, history_v1 = train_text_model(
    lstm_v1, txt_train_loader, txt_val_loader, criterion_rnn, optimizer_v1,
    scheduler_v1, DEVICE, epochs=5, patience=3, model_name='lstm_v1'
)

plot_training_curves(history_v1, title='LSTM v1: Single-Layer, Learned Embeddings')

In [ ]:
results_lstm_v1 = evaluate_text_and_report(
    lstm_v1, txt_test_loader, criterion_rnn, DEVICE, BAND_NAMES, 'LSTM v1 (Single, Learned)'
)

### 7.4 Variation B: Bidirectional Stacked LSTM with GloVe + Dropout

In [ ]:
lstm_v2 = LSTMClassifier(
    vocab_size=len(vocab), embed_dim=GLOVE_DIM if glove_matrix is not None else EMBED_DIM,
    hidden_dim=HIDDEN_DIM, num_classes=NUM_CLASSES, num_layers=2,
    bidirectional=True, dropout=0.4,
    pretrained_embeddings=glove_matrix
).to(DEVICE)

params_v2 = sum(p.numel() for p in lstm_v2.parameters())
print(f'LSTM v2 (BiLSTM, 2-layer, GloVe) — Parameters: {params_v2:,}')

optimizer_v2 = optim.Adam(lstm_v2.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler_v2 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_v2, mode='min', factor=0.5, patience=2)

lstm_v2, history_v2 = train_text_model(
    lstm_v2, txt_train_loader, txt_val_loader, criterion_rnn, optimizer_v2,
    scheduler_v2, DEVICE, epochs=5, patience=3, model_name='lstm_v2'
)

plot_training_curves(history_v2, title='LSTM v2: Bidirectional Stacked + GloVe')

In [ ]:
results_lstm_v2 = evaluate_text_and_report(
    lstm_v2, txt_test_loader, criterion_rnn, DEVICE, BAND_NAMES, 'LSTM v2 (BiLSTM, GloVe)'
)

### 7.5 RNN Comparison Discussion

In [ ]:
rnn_comparison = pd.DataFrame({
    'Model': ['LSTM v1 (Single, Learned)', 'LSTM v2 (BiLSTM, GloVe)'],
    'Test Accuracy': [results_lstm_v1['accuracy'], results_lstm_v2['accuracy']],
    'Test Macro F1': [results_lstm_v1['macro_f1'], results_lstm_v2['macro_f1']],
    'Test Loss': [results_lstm_v1['loss'], results_lstm_v2['loss']],
})
print(rnn_comparison.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(2)
width = 0.35
ax.bar(x - width/2, rnn_comparison['Test Accuracy'], width, label='Accuracy')
ax.bar(x + width/2, rnn_comparison['Test Macro F1'], width, label='Macro F1')
ax.set_xticks(x)
ax.set_xticklabels(rnn_comparison['Model'], fontsize=9)
ax.set_ylabel('Score')
ax.set_title('RNN: Single LSTM vs Bidirectional Stacked LSTM', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1)
for i, (acc, f1) in enumerate(zip(rnn_comparison['Test Accuracy'], rnn_comparison['Test Macro F1'])):
    ax.text(i - width/2, acc + 0.01, f'{acc:.3f}', ha='center', fontsize=9)
    ax.text(i + width/2, f1 + 0.01, f'{f1:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

**Discussion — RNN Comparison**

The bidirectional stacked LSTM with GloVe embeddings (v2) is expected to outperform the single-layer model (v1) for several reasons:

1. **Pre-trained embeddings**: GloVe vectors capture semantic relationships (e.g., "spacious" ≈ "roomy", "luxury" ≈ "premium") that the model doesn't need to learn from scratch, improving convergence speed and generalisation.
2. **Bidirectionality**: Reading the description in both directions allows the model to capture context from both preceding and following words, which is critical for real-estate language where adjectives and amenities can appear in any order.
3. **Stacked layers**: Two LSTM layers learn hierarchical temporal features — the first layer captures local phrases ("hardwood floors", "granite countertops") while the second layer captures document-level sentiment and value signals.
4. **Dropout regularisation**: Higher dropout (0.4) in the deeper model prevents overfitting despite increased capacity.

We select the best-performing LSTM variant as the RNN branch for downstream business integration.

## 8. Business Integration

### 8.1 Decision Rule

The combined business logic flags collateral risk by comparing the CNN-predicted visual band and the RNN-predicted text band against the applicant's *claimed* value (simulated as the ground-truth band):

- **HIGH** (manual appraisal): both model predictions are ≥2 bands below the claimed value.
- **MEDIUM** (secondary review): exactly one prediction is ≥2 bands below.
- **LOW** (auto-proceed): neither prediction diverges significantly from the claim.

In [ ]:
def collateral_risk_flag(pred_img_band, pred_txt_band, claimed_band):
    """Compute collateral risk flag from CNN and RNN predictions.
    
    Bands are 0-indexed integers (0=Budget ... 4=Luxury).
    'claimed_band' is the value the applicant claims (ground truth in our test set).
    """
    img_gap = claimed_band - pred_img_band
    txt_gap = claimed_band - pred_txt_band
    
    if img_gap >= 2 and txt_gap >= 2:
        return 'HIGH'
    elif img_gap >= 2 or txt_gap >= 2:
        return 'MEDIUM'
    else:
        return 'LOW'


# Since image and text test sets are different samples from the same domain,
# we demonstrate the business rule on synthetic pairings:
# Take the first min(N_img_test, N_txt_test) predictions from each branch
# and pair them randomly (both predict into the same 5-band space).

n_demo = min(len(results_cnn_transfer['preds']), len(results_lstm_v2['preds']), 200)

np.random.seed(RANDOM_STATE)
img_idx = np.random.choice(len(results_cnn_transfer['preds']), n_demo, replace=False)
txt_idx = np.random.choice(len(results_lstm_v2['preds']), n_demo, replace=False)

demo_img_preds = results_cnn_transfer['preds'][img_idx]
demo_txt_preds = results_lstm_v2['preds'][txt_idx]
demo_img_true = results_cnn_transfer['labels'][img_idx]
demo_txt_true = results_lstm_v2['labels'][txt_idx]

# Use image ground-truth as the "claimed" band (simulating what the applicant states)
claimed = demo_img_true

risk_flags = [collateral_risk_flag(ip, tp, c) for ip, tp, c in zip(demo_img_preds, demo_txt_preds, claimed)]

print('Risk flag distribution:')
print(pd.Series(risk_flags).value_counts())

In [ ]:
# Side-by-side table of 20 example predictions
demo_df = pd.DataFrame({
    'CNN Predicted': [BAND_NAMES[p] for p in demo_img_preds[:20]],
    'CNN True': [BAND_NAMES[t] for t in demo_img_true[:20]],
    'RNN Predicted': [BAND_NAMES[p] for p in demo_txt_preds[:20]],
    'RNN True': [BAND_NAMES[t] for t in demo_txt_true[:20]],
    'Claimed Band': [BAND_NAMES[c] for c in claimed[:20]],
    'Risk Flag': risk_flags[:20],
})

print('=== Business Integration: Sample Predictions (20 examples) ===')
print(demo_df.to_string(index=True))

In [ ]:
# Required comparison chart: CNN vs RNN vs Combined
# For the combined system, define "combined accuracy" as: the risk flag is correct
# where correct = LOW when both models agree with claimed, HIGH when both are far off, etc.

# Simpler metric: agreement rate (how often combined output aligns with what a human reviewer would flag)
# Here we compare each model's standalone F1 plus the combined system's ability to detect mismatches

best_cnn_model = 'ResNet-50 Transfer'
best_rnn_model = 'BiLSTM + GloVe'

fig, ax = plt.subplots(figsize=(10, 6))
models_names = ['CNN\n(ResNet-50)', 'RNN\n(BiLSTM)', 'Combined\nBusiness Rule']
accuracies = [results_cnn_transfer['accuracy'], results_lstm_v2['accuracy'], None]
f1s = [results_cnn_transfer['macro_f1'], results_lstm_v2['macro_f1'], None]

# Combined system accuracy: fraction of examples where at least one model is correct
both_correct = ((demo_img_preds == claimed) | (demo_txt_preds == claimed)).mean()
accuracies[2] = both_correct

# Combined "F1": for the 3-class risk flag, compute against an oracle
oracle_flags = [collateral_risk_flag(t1, t2, c) for t1, t2, c in zip(demo_img_true, demo_txt_true, claimed)]
combined_f1 = f1_score(
    oracle_flags, risk_flags, average='macro',
    labels=['LOW', 'MEDIUM', 'HIGH']
)
f1s[2] = combined_f1

x = np.arange(3)
width = 0.35
bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy / Coverage', color='steelblue')
bars2 = ax.bar(x + width/2, f1s, width, label='Macro F1', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(models_names)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison: CNN vs RNN vs Combined', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

**Business Integration Discussion**

The combined system provides value beyond either model alone:

- A property with a **good photo** (matching claimed band) but a **weak description** (lower predicted band) gets a `MEDIUM` flag, prompting the underwriter to investigate the text inconsistency — perhaps the listing has been embellished with a stock photo.
- A property where **both signals agree** with the claimed value gets a `LOW` flag, meaning the automated pipeline can fast-track it without human review — reducing operational costs.
- The `HIGH` flag catches properties where **both** data sources suggest the collateral is over-valued — the highest-risk scenario that demands a physical appraisal.

This tiered approach mirrors how real mortgage teams triage risk: automated pipelines handle the easy cases, and scarce human reviewers focus on genuinely risky ones.

## 9. Joint Model (+10 Bonus)

The joint model concatenates the CNN feature vector (from the global average pooling layer of ResNet-50, 2048-d) with the LSTM final hidden state (256-d for bidirectional) and passes them through fully connected layers to predict the price band.

For this, we need samples that have **both** an image and a text description with a shared price label. We construct a paired dataset from our existing data by combining entries.

In [ ]:
class JointModel(nn.Module):
    """Multimodal model: CNN features + LSTM features -> FC -> price band."""
    def __init__(self, cnn_backbone, lstm_model, cnn_feat_dim, lstm_feat_dim,
                 num_classes, dropout=0.4):
        super().__init__()
        self.cnn_backbone = cnn_backbone
        self.lstm_model = lstm_model
        
        combined_dim = cnn_feat_dim + lstm_feat_dim
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes),
        )
    
    def forward(self, image, text):
        # CNN features: use all layers except the final fc
        with torch.no_grad():
            cnn_feat = self._get_cnn_features(image)
        # LSTM features
        lstm_feat = self.lstm_model.get_features(text)
        # Concatenate and classify
        combined = torch.cat([cnn_feat, lstm_feat], dim=1)
        return self.classifier(combined)
    
    def _get_cnn_features(self, x):
        """Extract features before the final FC layer of ResNet-50."""
        model = self.cnn_backbone
        x = model.conv1(x)
        x = model.bn1(x)
        x = model.relu(x)
        x = model.maxpool(x)
        x = model.layer1(x)
        x = model.layer2(x)
        x = model.layer3(x)
        x = model.layer4(x)
        x = model.avgpool(x)
        x = torch.flatten(x, 1)
        return x

In [ ]:
class PairedDataset(Dataset):
    """Dataset for joint model: returns (image, text, label) tuples.
    
    Pairs are created by matching samples with the same price band.
    Each image sample is paired with a text sample from the same band.
    """
    def __init__(self, df_img, df_txt, desc_col, vocab, max_len, img_transform):
        self.img_transform = img_transform
        self.vocab = vocab
        self.max_len = max_len
        self.pairs = []
        
        for band_id in range(NUM_CLASSES):
            img_subset = df_img[df_img['price_band_id'] == band_id].reset_index(drop=True)
            txt_subset = df_txt[df_txt['price_band_id'] == band_id].reset_index(drop=True)
            n_pairs = min(len(img_subset), len(txt_subset))
            for i in range(n_pairs):
                self.pairs.append({
                    'image_path': img_subset.iloc[i]['image_path'],
                    'description': txt_subset.iloc[i][desc_col],
                    'label': band_id,
                })
        
        print(f'Paired dataset: {len(self.pairs)} samples')
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        item = self.pairs[idx]
        img = Image.open(item['image_path']).convert('RGB')
        if self.img_transform:
            img = self.img_transform(img)
        text = torch.tensor(self.vocab.encode(item['description'], self.max_len), dtype=torch.long)
        return img, text, int(item['label'])

In [ ]:
# Create paired datasets from training/val/test splits
paired_train = PairedDataset(df_img_train, df_txt_train, TEXT_DESC_COL, vocab, MAX_SEQ_LEN, train_transform)
paired_val = PairedDataset(df_img_val, df_txt_val, TEXT_DESC_COL, vocab, MAX_SEQ_LEN, val_test_transform)
paired_test = PairedDataset(df_img_test, df_txt_test, TEXT_DESC_COL, vocab, MAX_SEQ_LEN, val_test_transform)

paired_train_loader = DataLoader(paired_train, batch_size=BATCH_SIZE, shuffle=True,
                                 num_workers=NUM_WORKERS, pin_memory=True)
paired_val_loader = DataLoader(paired_val, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)
paired_test_loader = DataLoader(paired_test, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
# Build joint model from the best CNN and LSTM
best_lstm = lstm_v2  # bidirectional: hidden_dim * 2
lstm_feat_dim = HIDDEN_DIM * 2  # bidirectional
cnn_feat_dim = 2048  # ResNet-50 GAP output

joint_model = JointModel(
    cnn_backbone=cnn_transfer,
    lstm_model=best_lstm,
    cnn_feat_dim=cnn_feat_dim,
    lstm_feat_dim=lstm_feat_dim,
    num_classes=NUM_CLASSES,
    dropout=0.4
).to(DEVICE)

# Only train the joint classifier (freeze both backbones)
for param in joint_model.cnn_backbone.parameters():
    param.requires_grad = False
for param in joint_model.lstm_model.parameters():
    param.requires_grad = False

trainable_joint = sum(p.numel() for p in joint_model.parameters() if p.requires_grad)
print(f'Joint model — Trainable parameters: {trainable_joint:,}')

In [ ]:
# Joint model training loop
def train_joint_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, texts, labels in loader:
        images, texts = images.to(device), texts.to(device)
        labels = torch.tensor(labels, dtype=torch.long).to(device) if not isinstance(labels, torch.Tensor) else labels.to(device)
        optimizer.zero_grad()
        outputs = model(images, texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate_joint(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for images, texts, labels in loader:
        images, texts = images.to(device), texts.to(device)
        labels = torch.tensor(labels, dtype=torch.long).to(device) if not isinstance(labels, torch.Tensor) else labels.to(device)
        outputs = model(images, texts)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return running_loss / total, correct / total, np.array(all_preds), np.array(all_labels)


criterion_joint = nn.CrossEntropyLoss()
optimizer_joint = optim.Adam(joint_model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_joint = optim.lr_scheduler.ReduceLROnPlateau(optimizer_joint, mode='min', factor=0.5, patience=2)

history_joint = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_f1_joint = 0.0
patience_counter = 0
PATIENCE = 7

print('=== Training Joint Model ===')
for epoch in range(5):
    train_loss, train_acc = train_joint_epoch(joint_model, paired_train_loader, criterion_joint, optimizer_joint, DEVICE)
    val_loss, val_acc, val_preds, val_labels = evaluate_joint(joint_model, paired_val_loader, criterion_joint, DEVICE)
    val_f1 = f1_score(val_labels, val_preds, average='macro')
    
    history_joint['train_loss'].append(train_loss)
    history_joint['train_acc'].append(train_acc)
    history_joint['val_loss'].append(val_loss)
    history_joint['val_acc'].append(val_acc)
    
    scheduler_joint.step(val_loss)
    
    marker = ''
    if val_f1 > best_val_f1_joint:
        best_val_f1_joint = val_f1
        torch.save(joint_model.state_dict(), MODEL_DIR / 'joint_best.pt')
        patience_counter = 0
        marker = ' *'
    else:
        patience_counter += 1
    
    if (epoch + 1) % 2 == 0 or epoch == 0 or marker:
        print(f'Epoch {epoch+1:3d}/5 | Train loss: {train_loss:.4f} acc: {train_acc:.4f} | '
              f'Val loss: {val_loss:.4f} acc: {val_acc:.4f} F1: {val_f1:.4f}{marker}')
    
    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch+1}')
        break

joint_model.load_state_dict(torch.load(MODEL_DIR / 'joint_best.pt', weights_only=True))
plot_training_curves(history_joint, title='Joint Model (CNN + LSTM)')

In [ ]:
# Test evaluation — Joint model
test_loss_j, test_acc_j, preds_j, labels_j = evaluate_joint(
    joint_model, paired_test_loader, criterion_joint, DEVICE
)
test_f1_j = f1_score(labels_j, preds_j, average='macro')

print(f'\n=== Joint Model — Test Results ===')
print(f'Loss: {test_loss_j:.4f} | Accuracy: {test_acc_j:.4f} | Macro F1: {test_f1_j:.4f}')
print(f'\nClassification Report:')
print(classification_report(labels_j, preds_j, target_names=BAND_NAMES, digits=4))

cm = confusion_matrix(labels_j, preds_j)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=BAND_NAMES)
disp.plot(ax=ax, cmap='Greens', values_format='d')
ax.set_title('Joint Model — Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

results_joint = {'accuracy': test_acc_j, 'macro_f1': test_f1_j, 'loss': test_loss_j}

In [ ]:
# Final comparison: CNN-only vs RNN-only vs Joint
final_comparison = pd.DataFrame({
    'Model': ['CNN (ResNet-50)', 'RNN (BiLSTM)', 'Joint (CNN+LSTM)'],
    'Accuracy': [results_cnn_transfer['accuracy'], results_lstm_v2['accuracy'], results_joint['accuracy']],
    'Macro F1': [results_cnn_transfer['macro_f1'], results_lstm_v2['macro_f1'], results_joint['macro_f1']],
})

print('=== Final Model Comparison ===')
print(final_comparison.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(3)
width = 0.35
bars1 = ax.bar(x - width/2, final_comparison['Accuracy'], width, label='Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, final_comparison['Macro F1'], width, label='Macro F1', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(final_comparison['Model'])
ax.set_title('CNN-Only vs RNN-Only vs Joint Model', fontweight='bold')
ax.set_ylabel('Score')
ax.legend()
ax.set_ylim(0, 1)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

**Joint Model Discussion**

The joint model combines visual and textual features into a single prediction. By fusing the 2048-d CNN feature vector with the 256-d LSTM hidden state, the model can leverage complementary signals:

- The CNN captures **visual quality** (condition of the facade, landscaping, architectural style)
- The LSTM captures **textual quality** (amenity mentions, condition language, neighbourhood descriptors)

When these signals agree, the joint model should be more confident and accurate than either unimodal model. When they conflict, the fusion layers learn to weight the more reliable signal for each price band.

This architecture mirrors production multimodal systems where separate feature extractors are combined at the decision layer, balancing modularity (each branch can be updated independently) with integration power.

## 10. Explainability

### 10.1 Grad-CAM for CNN

In [ ]:
class GradCAM:
    """Gradient-weighted Class Activation Mapping for CNN interpretability."""
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)
    
    def _save_activation(self, module, input, output):
        self.activations = output.detach()
    
    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        input_tensor = input_tensor.requires_grad_(True)
        with torch.enable_grad():
            output = self.model(input_tensor)

            if target_class is None:
                target_class = output.argmax(dim=1).item()

            self.model.zero_grad()
            output[0, target_class].backward()
        
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, target_class


def show_gradcam(model, img_path, transform, target_layer, class_names, device):
    """Display original image alongside Grad-CAM overlay."""
    img_pil = Image.open(img_path).convert('RGB')
    img_tensor = transform(img_pil).unsqueeze(0).to(device)
    
    grad_cam = GradCAM(model, target_layer)
    cam, pred_class = grad_cam.generate(img_tensor)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original
    axes[0].imshow(img_pil.resize((IMG_SIZE, IMG_SIZE)))
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # Grad-CAM heatmap
    axes[1].imshow(cam, cmap='jet')
    axes[1].set_title('Grad-CAM Heatmap')
    axes[1].axis('off')
    
    # Overlay
    img_resized = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE))) / 255.0
    cam_colored = plt.cm.jet(cam)[:, :, :3]
    overlay = 0.5 * img_resized + 0.5 * cam_colored
    axes[2].imshow(overlay)
    axes[2].set_title(f'Overlay — Predicted: {class_names[pred_class]}')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Show Grad-CAM for 4 test images
target_layer = cnn_transfer.layer4[-1]
sample_indices = np.random.choice(len(df_img_test), 4, replace=False)

for idx in sample_indices:
    row = df_img_test.iloc[idx]
    print(f'True: {row["price_band"]} (${row["price"]:,.0f})')
    show_gradcam(cnn_transfer, row['image_path'], val_test_transform, target_layer, BAND_NAMES, DEVICE)

### 10.2 LSTM Attention / Token Importance

In [ ]:
def get_token_importance(model, text, vocab, max_len, device, class_names):
    """Compute token importance using input gradient method."""
    model.eval()
    tokens = tokenize(text)[:max_len]
    encoded = vocab.encode(text, max_len)
    input_tensor = torch.tensor([encoded], dtype=torch.long).to(device)

    with torch.enable_grad():
        embedding = model.embedding(input_tensor)
        embedding.requires_grad_(True)
        embedding.retain_grad()

        embedded = model.dropout(embedding)
        lengths = (input_tensor != 0).sum(dim=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.clamp(min=1), batch_first=True, enforce_sorted=False
        )
        _, (hidden, _) = model.lstm(packed)
        if model.bidirectional:
            hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            hidden = hidden[-1]
        out = model.dropout(hidden)
        logits = model.fc(out)

        pred_class = logits.argmax(dim=1).item()
        logits[0, pred_class].backward()

    grads = embedding.grad[0].detach().cpu()
    importance = torch.norm(grads[:len(tokens)], dim=1).numpy()
    importance = importance / (importance.max() + 1e-8)

    return tokens, importance, class_names[pred_class]


def visualize_token_importance(tokens, importance, predicted, title=''):
    """Display tokens coloured by importance."""
    fig, ax = plt.subplots(figsize=(max(14, len(tokens) * 0.3), 2))
    
    # Show top 40 tokens for readability
    n_show = min(40, len(tokens))
    tokens_show = tokens[:n_show]
    importance_show = importance[:n_show]
    
    colors = plt.cm.Reds(importance_show)
    for i, (token, imp) in enumerate(zip(tokens_show, importance_show)):
        ax.text(i, 0, token, fontsize=9, ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=plt.cm.Reds(imp), alpha=0.7))
    
    ax.set_xlim(-1, n_show)
    ax.set_ylim(-0.5, 0.5)
    ax.axis('off')
    ax.set_title(f'{title} — Predicted: {predicted}', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Show token importance for 4 test descriptions
best_lstm_for_explain = lstm_v2
sample_txt_indices = np.random.choice(len(df_txt_test), 4, replace=False)

for idx in sample_txt_indices:
    row = df_txt_test.iloc[idx]
    desc = row[TEXT_DESC_COL]
    tokens, importance, predicted = get_token_importance(
        best_lstm_for_explain, desc, vocab, MAX_SEQ_LEN, DEVICE, BAND_NAMES
    )
    print(f'\nTrue: {row["price_band"]} | Text: {desc[:150]}...')
    visualize_token_importance(tokens, importance, predicted, title=f'True: {row["price_band"]}')

## 11. Deployment

We deploy the pipeline as a **Streamlit** web application with three panels:
1. **Image panel** — upload a property photo, get CNN prediction + Grad-CAM
2. **Text panel** — paste a listing description, get RNN prediction + highlighted tokens
3. **Combined panel** — see the collateral risk flag (LOW/MEDIUM/HIGH)

The deployment code is saved as `app.py` and can be run with `streamlit run app.py`.

In [ ]:
# Save model weights and vocabulary for deployment
torch.save(cnn_transfer.state_dict(), MODEL_DIR / 'cnn_transfer_deploy.pt')
torch.save(best_lstm.state_dict(), MODEL_DIR / 'lstm_deploy.pt')

# Save vocabulary
import json
with open(MODEL_DIR / 'vocab.json', 'w') as f:
    json.dump({
        'word2idx': vocab.word2idx,
        'max_seq_len': MAX_SEQ_LEN,
        'embed_dim': GLOVE_DIM if glove_matrix is not None else EMBED_DIM,
        'hidden_dim': HIDDEN_DIM,
        'num_classes': NUM_CLASSES,
        'band_names': BAND_NAMES,
    }, f)

print(f'Saved deployment artefacts to {MODEL_DIR}/')
for p in MODEL_DIR.iterdir():
    print(f'  {p.name}: {p.stat().st_size / 1e6:.1f} MB')

In [ ]:
# The Streamlit app is saved as app.py in the project root.
# It loads the trained CNN and LSTM models from models/ and provides:
#   - Image upload panel with CNN prediction
#   - Text input panel with RNN prediction
#   - Combined collateral risk flag (LOW / MEDIUM / HIGH)

print('Streamlit app: app.py (already created in the project root)')
print('Run with: streamlit run app.py')

# The full Streamlit code is shown below for reference:
STREAMLIT_REFERENCE = '''
"""Mortgage Collateral Risk Assistant — Streamlit Deployment."""
import streamlit as st
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
from PIL import Image
import json
import re
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path

MODEL_DIR = Path("models")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224


def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text.split()


class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=2, bidirectional=True, dropout=0.4):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                           batch_first=True, bidirectional=bidirectional,
                           dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * self.num_directions, num_classes)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        lengths = (x != 0).sum(dim=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.clamp(min=1), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        if self.bidirectional:
            hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            hidden = hidden[-1]
        return self.fc(self.dropout(hidden))


@st.cache_resource
def load_models():
    with open(MODEL_DIR / "vocab.json") as f:
        config = json.load(f)

    cnn = models.resnet50(weights=None)
    cnn.fc = nn.Sequential(
        nn.Linear(2048, 256), nn.ReLU(True), nn.Dropout(0.4),
        nn.Linear(256, config["num_classes"]))
    cnn.load_state_dict(torch.load(MODEL_DIR / "cnn_transfer_deploy.pt",
                                   map_location=DEVICE, weights_only=True))
    cnn.eval().to(DEVICE)

    lstm = LSTMClassifier(
        len(config["word2idx"]), config["embed_dim"],
        config["hidden_dim"], config["num_classes"])
    lstm.load_state_dict(torch.load(MODEL_DIR / "lstm_deploy.pt",
                                    map_location=DEVICE, weights_only=True))
    lstm.eval().to(DEVICE)

    return cnn, lstm, config


def predict_image(cnn, img_pil, config):
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    tensor = transform(img_pil.convert("RGB")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = cnn(tensor)
    probs = F.softmax(logits, dim=1).cpu().numpy()[0]
    pred = int(probs.argmax())
    return pred, probs


def predict_text(lstm, text, config):
    word2idx = config["word2idx"]
    max_len = config["max_seq_len"]
    tokens = tokenize(text)[:max_len]
    indices = [word2idx.get(t, word2idx.get("<UNK>", 1)) for t in tokens]
    indices += [0] * (max_len - len(indices))
    tensor = torch.tensor([indices], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        logits = lstm(tensor)
    probs = F.softmax(logits, dim=1).cpu().numpy()[0]
    pred = int(probs.argmax())
    return pred, probs


def risk_flag(img_band, txt_band, claimed_band):
    ig = claimed_band - img_band
    tg = claimed_band - txt_band
    if ig >= 2 and tg >= 2:
        return "HIGH"
    elif ig >= 2 or tg >= 2:
        return "MEDIUM"
    return "LOW"


st.set_page_config(page_title="Mortgage Collateral Risk", layout="wide")
st.title("Mortgage Collateral Risk Assistant")
st.markdown("Upload a property photo and paste its listing description to assess collateral risk.")

cnn, lstm, config = load_models()
bands = config["band_names"]

col1, col2 = st.columns(2)

with col1:
    st.subheader("Property Photo")
    uploaded = st.file_uploader("Upload image", type=["jpg", "jpeg", "png"])
    img_pred, img_probs = None, None
    if uploaded:
        img = Image.open(uploaded)
        st.image(img, use_container_width=True)
        img_pred, img_probs = predict_image(cnn, img, config)
        st.metric("CNN Prediction", bands[img_pred], f"{img_probs[img_pred]*100:.1f}%")

with col2:
    st.subheader("Listing Description")
    text_input = st.text_area("Paste description", height=200)
    txt_pred, txt_probs = None, None
    if text_input.strip():
        txt_pred, txt_probs = predict_text(lstm, text_input, config)
        st.metric("RNN Prediction", bands[txt_pred], f"{txt_probs[txt_pred]*100:.1f}%")

st.divider()
if img_pred is not None and txt_pred is not None:
    claimed = st.selectbox("Applicant\'s claimed value tier", bands, index=2)
    claimed_id = bands.index(claimed)
    flag = risk_flag(img_pred, txt_pred, claimed_id)
    colour = {"LOW": "green", "MEDIUM": "orange", "HIGH": "red"}[flag]
    st.markdown(f"### Collateral Risk: :{colour}[{flag}]")
    if flag == "LOW":
        st.success("Both models support the claimed value. Auto-proceed recommended.")
    elif flag == "MEDIUM":
        st.warning("One model diverges from the claimed value. Secondary review recommended.")
    else:
        st.error("Both models suggest the collateral is over-valued. Manual appraisal required.")
'''
print('(See app.py for the complete runnable Streamlit code.)')

### Deployment Screenshot

The Streamlit prototype provides:
- **Left panel**: Image upload with CNN prediction and confidence score
- **Right panel**: Text area for listing description with RNN prediction and confidence score
- **Bottom panel**: Combined collateral risk flag (LOW/MEDIUM/HIGH) with colour-coded recommendation

To deploy on Hugging Face Spaces:
1. Create a new Space with the Streamlit SDK
2. Upload `app.py`, `models/` directory (weights + vocab), and `requirements.txt`
3. The app will be publicly accessible

*Include a screen recording or screenshot of the working prototype here after running locally.*

## 12. Business Framing, Ethics, and Limitations

### Business Framing

**What business decision does this pipeline support?**

This pipeline automates the initial triage of mortgage collateral assessments. When a borrower submits a property listing (photo + description) as collateral for a loan, the system evaluates whether the claimed property value is consistent with visual and textual evidence. This supports three workflow paths:

| Risk Flag | Action | Volume (expected) |
|-----------|--------|-------------------|
| LOW | Auto-proceed to next underwriting stage | ~60-70% of applications |
| MEDIUM | Route to junior reviewer for secondary check | ~20-25% |
| HIGH | Mandate physical appraisal before approval | ~5-15% |

**Who is the end user?** Mortgage underwriters, risk officers, and automated loan processing systems at banks and P2P lending platforms.

### Cost of Errors

| Error Type | Impact | Estimated Cost |
|------------|--------|----------------|
| **False Positive** (flagged as HIGH but property is fine) | Unnecessary manual appraisal ordered | $300-500 per appraisal + processing delays |
| **False Negative** (flagged as LOW but property is overvalued) | Loan issued against inflated collateral | Potential loss of 10-50% of loan value in default |

The asymmetric cost structure means we should tune the system towards **higher recall for HIGH-risk cases** (accept more false positives to avoid expensive false negatives).

### Ethical Considerations

1. **Location bias in images**: The CNN may learn to associate certain neighbourhood aesthetics (e.g., greenery, paved roads) with higher value, potentially disadvantaging properties in lower-income areas that are structurally sound but visually distinct.

2. **Description style bias**: The LSTM may penalise listings written in less polished English (e.g., by non-native speakers or FSBO sellers), conflating writing quality with property quality.

3. **Demographic proxy effects**: Property photos may inadvertently encode information about the demographics of a neighbourhood (religious buildings, cultural signage), creating a proxy for protected characteristics under fair lending laws.

4. **Feedback loops**: If the system systematically under-values properties in certain areas, it may reduce lending in those areas, further depressing property values — a self-reinforcing cycle.

**Mitigation strategies:**
- Regular fairness audits across geographic segments
- Human review for all HIGH-risk decisions (no fully automated rejections)
- Description quality normalisation (focus on content signals, not writing style)
- Diverse training data spanning property types, regions, and price ranges

### Limitations

- Image and text datasets are from different real-estate markets (US vs UK), limiting direct cross-domain pairing for the joint model
- Price band boundaries are dataset-specific; deploying across markets requires recalibration
- Single exterior photos capture limited information (interior condition, structural issues are invisible)
- Text descriptions are seller-authored and inherently biased towards positive framing

## 13. Contribution Table & References

### Team Contribution

| Member | Sections | Contribution |
|--------|----------|-------------|
| Raivo Strods | All sections 1-12 | 100% — individual project |

### References

#### Deep Learning Foundations

1. He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep Residual Learning for Image Recognition. *CVPR 2016*.
2. Hochreiter, S., & Schmidhuber, J. (1997). Long Short-Term Memory. *Neural Computation*, 9(8), 1735–1780.
3. Selvaraju, R. R., et al. (2017). Grad-CAM: Visual Explanations from Deep Neural Networks via Gradient-based Localization. *ICCV 2017*.
4. Pennington, J., Socher, R., & Manning, C. (2014). GloVe: Global Vectors for Word Representation. *EMNLP 2014*.

#### CNN-LSTM Hybrid Models for Risk Assessment

5. A hybrid model based on CNN-LSTM for assessing the risk of increasing claims in insurance companies. *PeerJ Computer Science*, 2025. https://doi.org/10.7717/peerj-cs.2830
6. Yao, J., Wang, J., Wang, B., Liu, B., & Jiang, M. (2024). A Hybrid CNN-LSTM Model for Enhancing Bond Default Risk Prediction. *Journal of Computer Technology and Software*, 3(6). https://doi.org/10.5281/zenodo.13910344
7. The Evaluation on the Credit Risk of Enterprises with the CNN-LSTM-ATT Model. *Computational Intelligence and Neuroscience*, 2022, 6826573. https://doi.org/10.1155/2022/6826573

#### Datasets

8. House Prices and Images — SoCal. Kaggle Dataset by ted8080. https://www.kaggle.com/datasets/ted8080/house-prices-and-images-socal — Original author repo: https://github.com/tncy67/House-Price-Prediction-via-Computer-Vision
9. Real Estate Data London 2024. Kaggle Dataset by kanchana1990. https://www.kaggle.com/datasets/kanchana1990/real-estate-data-london-2024
10. Airbnb Open Data. Kaggle Dataset by arianazmoudeh. https://www.kaggle.com/datasets/arianazmoudeh/airbnbopendata

#### Tools & Frameworks

11. PyTorch Documentation. https://pytorch.org/docs/
12. Streamlit Documentation. https://docs.streamlit.io/

#### Additional Context

- Housing in London 2024 Report (2nd edition). Greater London Authority. https://data.london.gov.uk/download/24rpx/8cdbb084-982c-44f3-a890-765f4002cbaa/Housing%20in%20London%202024%20report%20-%202nd%20edition.pdf
- London properties price CatBoost+SHAP. Kaggle Notebook by dima806. https://www.kaggle.com/code/dima806/london-properties-price-catboost-shap
- Full reference list with local academic PDFs: see `docs/REFERENCES.md`